# 02 · Create training examples by breaking code

**Goal:** turn each compiling seed into several *broken* versions with real compiler errors.
Each example = **input:** broken code + compiler errors → **output:** the original, correct code.

**Rules:** every mistake must be fixable from the information given; mistakes are never placed inside strings or comments;
only versions that really fail to compile are kept; everything is seeded, so re-running gives the same data.

## 1. Setup
Load the seeds from notebook 01.

In [1]:
# Cell 3: Setup.
import sys                                              # To add the project folder to Python's import path
from pathlib import Path                                # File paths that work on any OS

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # Project folder, wherever it lives
sys.path.insert(0, str(ROOT))                           # So our .py files can be imported

import difflib                                          # Show what changed between two versions
from collections import Counter                         # Count things
import pandas as pd                                     # Tables
from csharp import compile_many                         # Our parallel build tool
from mutations import make_mutations, OPERATORS         # The ways we break code

SEEDS_FILE = ROOT / "data" / "processed" / "seeds.jsonl"         # Input: compiling programs
EXAMPLES_FILE = ROOT / "data" / "processed" / "examples.jsonl"   # Output: training examples
MAX_EXAMPLE_CHARS = 6_000                               # broken + errors + fixed must fit a small model's context (~1,700 tokens)

seeds = pd.read_json(SEEDS_FILE, lines=True)            # One row per seed
print(f"{len(seeds)} seeds from {seeds['group'].nunique()} groups | {len(OPERATORS)} operators")

367 seeds from 127 groups | 9 operators


## 2. Preview the mistakes on one seed
Each line shows the operator and the lines it changed (- original, + broken).

In [2]:
# Cell 5: Show every mutation of one seed as a mini diff.
def changed_lines(a, b):
    """Only the removed (-) and added (+) lines between two versions."""
    diff = difflib.unified_diff(a.splitlines(), b.splitlines(), lineterm="", n=0)   # n=0: no context lines
    return [line for line in diff if line[:1] in "+-" and not line.startswith(("+++", "---"))]

sample = seeds.iloc[5]                                  # Any seed will do
print("Seed:", sample["id"], sample["path"], "\n")
for m in make_mutations(sample["id"], sample["code"]):  # All broken versions of this seed
    print("+".join(m["kinds"]))                         # Which operator(s)
    for line in changed_lines(sample["code"], m["broken"])[:4]:
        print("   ", line.strip()[:110])                # The changed lines (shortened)

Seed: seed_0005 AspNetCore.Docs/aspnetcore/fundamentals/app-state/samples/6.0/SessionSample/Program.cs#v4 

drop_semicolon
    -var app = builder.Build();
    +var app = builder.Build()
drop_closing
    -    app.UseExceptionHandler("/Error");
    +    app.UseExceptionHandler("/Error";
wrong_member
    -builder.Services.AddSession();
    +builder.ServicesAsync.AddSession();
wrong_type
    -var builder = WebApplication.CreateBuilder(args);
    +var builder = webApplication.CreateBuilder(args);
wrong_local
    -builder.Services.AddRazorPages()
    +buillder.Services.AddRazorPages()


## 3. Generate, compile and keep only real failures
About 11 candidates per seed, so roughly 4,000 builds (about an hour with 4 parallel sandboxes).

In [3]:
# Cell 7: Make all mutations, compile them, keep the ones that fail, and save the training examples.
candidates = []                                         # One row per broken candidate
for s in seeds.to_dict("records"):                      # Every seed
    for m in make_mutations(s["id"], s["code"]):        # Every broken version of it
        candidates.append({"seed_id": s["id"], "group": s["group"], "kinds": "+".join(m["kinds"]),
                           "broken": m["broken"], "fixed": s["code"]})   # fixed = the original code
print(f"Candidates: {len(candidates):,}")

results = compile_many([c["broken"] for c in candidates], workers=4)   # Compile every broken version

examples = []
for c, r in zip(candidates, results):                   # Candidate and its build result, side by side
    c["fails"] = not r.succeeded                        # Remember whether it really broke (for the stats below)
    if r.succeeded:                                     # Still compiles: the mutation was harmless, skip it
        continue
    errors = r.error_text(c["broken"])                  # Error messages + offending lines (max 5)
    if len(c["broken"]) + len(errors) + len(c["fixed"]) > MAX_EXAMPLE_CHARS:   # Too long for the model
        continue
    examples.append({**{k: c[k] for k in ("seed_id", "group", "kinds", "broken", "fixed")},
                     "errors": errors,                  # What the model will see
                     "error_codes": [e["code"] for e in r.errors],   # For statistics
                     "n_errors": len(r.errors)})

ex = pd.DataFrame(examples)                             # One row per training example
ex.insert(0, "id", [f"ex_{i:05d}" for i in range(len(ex))])   # Stable IDs: ex_00000, ex_00001, ...
ex.to_json(EXAMPLES_FILE, orient="records", lines=True, force_ascii=False)   # Save as JSONL
print(f"Examples kept: {len(ex):,} (from {ex['seed_id'].nunique()} seeds, {ex['group'].nunique()} groups)")
print("Saved:", EXAMPLES_FILE)

Candidates: 2,744


100%|██████████| 2744/2744 [42:15<00:00,  1.08it/s]


Examples kept: 2,692 (from 367 seeds, 127 groups)
Saved: d:\AI Projects\csharp-fix-lora\data\processed\examples.jsonl


## 4. Statistics
How often each operator really broke the build, which errors we produced, and how long the examples are.

In [4]:
# Cell 9: Statistics.
cand = pd.DataFrame(candidates)                         # All candidates (with the "fails" flag)
by_kind = cand.groupby("kinds")["fails"].agg(["count", "mean"])   # Per operator: tried, and share that failed
by_kind.columns = ["tried", "broke the build"]
by_kind["broke the build"] = (by_kind["broke the build"] * 100).round(0).astype(int).astype(str) + "%"
singles = by_kind[~by_kind.index.str.contains(r"\+")]   # Single-mistake operators only (doubles are many combinations)
print(singles.sort_values("tried", ascending=False), "\n")
print("Two-mistake examples:", ex["kinds"].str.contains(r"\+").sum())

codes = Counter(c for codes in ex["error_codes"] for c in codes[:1])   # First error code of each example
print("\nMost common first error:", codes.most_common(10))
total = ex["broken"].str.len() + ex["errors"].str.len() + ex["fixed"].str.len()   # Example size in characters
print(f"\nExample size: median {total.median():,.0f} chars, max {total.max():,} (~{total.max() / 3.5:,.0f} tokens)")

                tried broke the build
kinds                                
drop_closing      367            100%
wrong_type        367             98%
drop_semicolon    367            100%
wrong_member      367             97%
wrong_local       367            100%
wrong_literal     231             95%
remove_using      135             86%
drop_new          109            100%
drop_async         58             98% 

Two-mistake examples: 376

Most common first error: [('CS0103', 848), ('CS1026', 395), ('CS1002', 356), ('CS1061', 311), ('CS0246', 166), ('CS1003', 140), ('CS0117', 76), ('CS1513', 62), ('CS1955', 60), ('CS4034', 51)]

Example size: median 1,491 chars, max 5,880 (~1,680 tokens)


## 5. One complete example
Exactly what the model will be trained on: the input (broken code + errors) and the expected output (fixed code).

In [5]:
# Cell 11: Show one training example in full.
e = ex.sample(1, random_state=7).iloc[0]                # A random (but repeatable) example
print("Kind:", e["kinds"], "| errors:", e["error_codes"])
print("\n----- COMPILER ERRORS (input) -----\n" + e["errors"])
print("\n----- WHAT CHANGED (- correct, + broken) -----")
for line in changed_lines(e["fixed"], e["broken"]):
    print("   ", line.strip()[:110])

Kind: wrong_literal | errors: ['CS0029', 'CS1662']

----- COMPILER ERRORS (input) -----
Line 9: error CS0029: Cannot implicitly convert type 'string' to 'bool'
    > options.CheckConsentNeeded = context => "true";
Line 9: error CS1662: Cannot convert lambda expression to intended delegate type because some of the return types in the block are not implicitly convertible to the delegate return type
    > options.CheckConsentNeeded = context => "true";

----- WHAT CHANGED (- correct, + broken) -----
    -    options.CheckConsentNeeded = context => true;
    +    options.CheckConsentNeeded = context => "true";
